наилушая модель на данный момент

In [ ]:
# ============================================================
# FINAL HYBRID MODEL v10
# ETS + XGBoost + Analog Ensemble + Weather API
#
# IMPROVEMENTS vs v9:
#
# ✅ optimized for WAPE
# ✅ stronger analog ensemble
# ✅ trend correction
# ✅ weekday analog blending
# ✅ holiday uplift correction
# ✅ residual momentum
# ✅ better morning-hour handling
# ✅ robust clipping
# ✅ train leakage fixes
# ✅ adaptive ensemble weights
# ✅ stronger slot features
#
# MAIN METRIC: WAPE
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import os
import ssl
import json
import urllib.request

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit

from xgboost import XGBRegressor
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# ============================================================
# CONFIG
# ============================================================

WORKING_HOURS = list(range(7, 23))
HOURS_PER_DAY = 16

LAT = 55.800
LON = 37.529

DATA_DIR = "./data"

FORECAST_START = "2026-04-27"
FORECAST_END   = "2026-05-03"

CUT_DATE = "2022-09-01"

HIGH_ERROR_HOURS = {7,8,9,10}

# ============================================================
# METRIC
# ============================================================

def wape(y_true, y_pred):

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    return (
        np.sum(np.abs(y_true - y_pred))
        /
        (np.sum(np.abs(y_true)) + 1e-9)
    ) * 100


# ============================================================
# HOLIDAYS
# ============================================================

_STATIC_HOLIDAYS_FIXED = {
    (1,1),(1,2),(1,3),(1,4),(1,5),(1,6),(1,7),(1,8),
    (2,23),(3,8),(5,1),(5,9),(6,12),(11,4),
}

_STATIC_EXTRA = {
    "2019-05-02","2019-05-03","2019-05-10",
    "2020-03-09",
    "2021-02-22","2021-05-10",
    "2022-03-07","2022-05-02","2022-05-10",
    "2023-02-24","2023-05-08",
    "2024-04-29","2024-04-30","2024-05-10",
    "2025-04-30","2025-05-02",
    "2026-03-09","2026-05-04",
}

# ============================================================
# WEATHER FALLBACK
# ============================================================

_CLIMATE_FALLBACK = {
    1:{"t":-7.0,"p":40},
    2:{"t":-6.0,"p":35},
    3:{"t":-0.5,"p":35},
    4:{"t":8.0,"p":40},
    5:{"t":15.0,"p":50},
    6:{"t":19.0,"p":70},
    7:{"t":21.5,"p":75},
    8:{"t":20.0,"p":70},
    9:{"t":14.0,"p":55},
    10:{"t":6.5,"p":55},
    11:{"t":-0.5,"p":45},
    12:{"t":-5.0,"p":45},
}

# ============================================================
# HTTP
# ============================================================

def _http_get(url, timeout=10):

    ctx = ssl.create_default_context()
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE

    try:

        req = urllib.request.Request(
            url,
            headers={"User-Agent":"forecast-model"}
        )

        with urllib.request.urlopen(
            req,
            timeout=timeout,
            context=ctx
        ) as r:

            return json.loads(r.read())

    except:
        return None


# ============================================================
# HOLIDAYS API
# ============================================================

def fetch_holidays_api(years):

    holiday_dates = set()

    for year in years:

        url = f"https://date.nager.at/api/v3/PublicHolidays/{year}/RU"

        data = _http_get(url)

        if data:

            for item in data:
                holiday_dates.add(item["date"])

        else:

            for m,d in _STATIC_HOLIDAYS_FIXED:

                holiday_dates.add(
                    f"{year}-{m:02d}-{d:02d}"
                )

    holiday_dates |= _STATIC_EXTRA

    return holiday_dates


# ============================================================
# WEATHER API
# ============================================================

def fetch_weather_api(start_date, end_date):

    url = (
        "https://archive-api.open-meteo.com/v1/archive"
        f"?latitude={LAT}"
        f"&longitude={LON}"
        f"&start_date={start_date}"
        f"&end_date={end_date}"
        "&hourly=temperature_2m,precipitation"
        "&timezone=Europe%2FMoscow"
    )

    data = _http_get(url)

    if data is None or "hourly" not in data:

        idx = pd.date_range(start_date, end_date, freq="h")

        df = pd.DataFrame(index=idx)

        df["temp_real"] = [
            _CLIMATE_FALLBACK[d.month]["t"]
            for d in idx
        ]

        df["precip_mm"] = 0.0

        return df

    h = data["hourly"]

    df = pd.DataFrame({
        "datetime": pd.to_datetime(h["time"]),
        "temp_real": h["temperature_2m"],
        "precip_mm": h["precipitation"],
    })

    df = df.set_index("datetime")

    return df


# ============================================================
# INDEX
# ============================================================

def make_working_index(start_date, end_date):

    days = pd.date_range(start_date, end_date, freq="D")

    idx = []

    for d in days:

        for h in WORKING_HOURS:

            idx.append(
                pd.Timestamp(d)
                +
                pd.Timedelta(hours=h)
            )

    return pd.DatetimeIndex(idx)


# ============================================================
# FEATURES
# ============================================================

def compute_features(
    df,
    holiday_set,
    weather_df,
    train_index=None,
):

    df = df.copy()

    # ========================================================
    # TIME
    # ========================================================

    df["hour"] = df.index.hour
    df["dayofweek"] = df.index.dayofweek
    df["month"] = df.index.month
    df["quarter"] = df.index.quarter

    df["is_weekend"] = (
        df["dayofweek"] >= 5
    ).astype(int)

    # ========================================================
    # CYCLICAL
    # ========================================================

    df["hour_sin"] = np.sin(
        2*np.pi*df["hour"]/24
    )

    df["hour_cos"] = np.cos(
        2*np.pi*df["hour"]/24
    )

    df["dow_sin"] = np.sin(
        2*np.pi*df["dayofweek"]/7
    )

    df["dow_cos"] = np.cos(
        2*np.pi*df["dayofweek"]/7
    )

    # ========================================================
    # BUSINESS FEATURES
    # ========================================================

    df["is_morning"] = (
        df["hour"] <= 10
    ).astype(int)

    df["is_lunch"] = (
        df["hour"].isin([11,12,13,14])
    ).astype(int)

    df["is_dinner"] = (
        df["hour"].isin([17,18,19,20])
    ).astype(int)

    # ========================================================
    # HOLIDAYS
    # ========================================================

    df["is_holiday"] = df.index.normalize().map(
        lambda x: int(
            x.strftime("%Y-%m-%d") in holiday_set
        )
    )

    df["is_pre_holiday"] = (
        df.index.normalize() + pd.Timedelta(days=1)
    ).map(
        lambda x: int(
            x.strftime("%Y-%m-%d") in holiday_set
        )
    )

    df["is_post_holiday"] = (
        df.index.normalize() - pd.Timedelta(days=1)
    ).map(
        lambda x: int(
            x.strftime("%Y-%m-%d") in holiday_set
        )
    )

    # ========================================================
    # WEATHER
    # ========================================================

    if weather_df is not None:

        df = df.join(
            weather_df,
            how="left"
        )

    temp_fallback = pd.Series(
        [_CLIMATE_FALLBACK[m]["t"] for m in df.index.month],
        index=df.index
    )

    df["temp_real"] = df["temp_real"].fillna(
        temp_fallback
    )

    df["precip_mm"] = df["precip_mm"].fillna(0)

    df["is_rainy"] = (
        df["precip_mm"] > 0.5
    ).astype(int)

    # ========================================================
    # LAGS
    # ========================================================

    for lag in [1,2,16,32,48,80,112,224]:

        df[f"lag_{lag}"] = (
            df["guests_count"]
            .shift(lag)
        )

    # ========================================================
    # ROLLING
    # ========================================================

    base = df["guests_count"].shift(1)

    for w in [16,32,80,112]:

        df[f"rolling_mean_{w}"] = (
            base.rolling(w).mean()
        )

        df[f"rolling_std_{w}"] = (
            base.rolling(w).std()
        )

    # ========================================================
    # SLOT FEATURES
    # ========================================================

    slot_ewm4 = []
    slot_ewm8 = []
    slot_roll2 = []
    slot_roll4 = []

    for (h,d), grp in df.groupby(["hour","dayofweek"]):

        g = grp.sort_index()["guests_count"].shift(1)

        e4 = g.ewm(span=4).mean()
        e8 = g.ewm(span=8).mean()

        slot_ewm4.append(e4)
        slot_ewm8.append(e8)

        slot_roll2.append(
            g.rolling(2).mean()
        )

        slot_roll4.append(
            g.rolling(4).mean()
        )

    df["slot_ewm4"] = (
        pd.concat(slot_ewm4).sort_index()
    )

    df["slot_ewm8"] = (
        pd.concat(slot_ewm8).sort_index()
    )

    df["slot_roll2"] = (
        pd.concat(slot_roll2).sort_index()
    )

    df["slot_roll4"] = (
        pd.concat(slot_roll4).sort_index()
    )

    # ========================================================
    # NEW FEATURES
    # ========================================================

    df["slot_momentum"] = (
        df["slot_ewm4"]
        -
        df["slot_ewm8"]
    )

    df["slot_acceleration"] = (
        df["slot_roll2"]
        -
        df["slot_roll4"]
    )

    df["lag1_vs_slot"] = (
        df["lag_1"]
        /
        (df["slot_ewm4"] + 1e-9)
    )

    # ========================================================
    # MEAN ENCODING
    # ========================================================

    if train_index is not None:

        tr = df.loc[
            df.index.isin(train_index)
            &
            df["guests_count"].notna()
        ]

        me_hd = tr.groupby(
            ["hour","dayofweek"]
        )["guests_count"].mean()

        me_h = tr.groupby(
            ["hour"]
        )["guests_count"].mean()

        df = df.join(
            me_hd.rename("me_hour_dow"),
            on=["hour","dayofweek"]
        )

        df = df.join(
            me_h.rename("me_hour"),
            on=["hour"]
        )

    else:

        df["me_hour_dow"] = 0
        df["me_hour"] = 0

    return df


# ============================================================
# ETS
# ============================================================

def prepare_ets(train_raw):

    df = train_raw.copy()

    df["datetime"] = (
        df["sale_date"]
        +
        pd.to_timedelta(df["sale_hour"], unit="h")
    )

    df = df.set_index("datetime")

    wi = make_working_index(
        df.index.min().date(),
        df.index.max().date()
    )

    df = df.reindex(wi)

    df["guests_count"] = (
        df["guests_count"]
        .interpolate(method="time")
    )

    train_ets = df["guests_count"]

    model = ExponentialSmoothing(
        train_ets,
        trend="add",
        seasonal="add",
        seasonal_periods=16,
        damped_trend=True,
    )

    fit = model.fit(optimized=True)

    return df, train_ets, fit


# ============================================================
# MODEL
# ============================================================

def build_model(
    train_raw,
    holiday_set,
    weather_df,
):

    df_working, train_ets, ets_fit = prepare_ets(
        train_raw
    )

    forecast_index = make_working_index(
        FORECAST_START,
        FORECAST_END
    )

    future_df = pd.DataFrame({
        "guests_count": np.nan
    }, index=forecast_index)

    full_df = pd.concat([
        df_working,
        future_df
    ])

    full_fe = compute_features(
        full_df,
        holiday_set,
        weather_df,
        train_index=df_working.index,
    )

    forecast_fe = full_fe.loc[
        full_fe.index.isin(forecast_index)
    ].copy()

    EXCLUDE = {
        "guests_count",
        "sale_hour",
        "sale_date",
        "datetime",
    }

    feature_cols = [
        c
        for c in full_fe.select_dtypes(include=[np.number]).columns
        if c not in EXCLUDE
    ]

    scaler = StandardScaler()

    scaler.fit(
        full_fe[feature_cols].fillna(0)
    )

    full_fe_sc = full_fe.copy()

    full_fe_sc[feature_cols] = scaler.transform(
        full_fe[feature_cols].fillna(0)
    )

    train_fe = full_fe_sc.loc[
        full_fe_sc.index.isin(df_working.index)
    ].copy()

    forecast_fe = full_fe_sc.loc[
        full_fe_sc.index.isin(forecast_index)
    ].copy()

    ets_fitted = ets_fit.fittedvalues

    common_idx = train_fe.index.intersection(
        ets_fitted.index
    )

    boost_df = train_fe.loc[common_idx].copy()

    boost_df["ets_residual"] = (
        df_working.loc[common_idx,"guests_count"]
        -
        ets_fitted.loc[common_idx]
    )

    for lag in [1,2,16,32,48,80,112]:

        boost_df[f"res_lag_{lag}"] = (
            boost_df["ets_residual"]
            .shift(lag)
        )

    boost_df["residual_momentum"] = (
        boost_df["res_lag_1"]
        -
        boost_df["res_lag_16"]
    )

    boost_df = boost_df.dropna()

    res_lag_cols = [
        c
        for c in boost_df.columns
        if c.startswith("res_lag")
    ]

    res_lag_cols.append(
        "residual_momentum"
    )

    res_scaler = StandardScaler()

    boost_df[res_lag_cols] = res_scaler.fit_transform(
        boost_df[res_lag_cols]
    )

    xgb_features = feature_cols + res_lag_cols

    X = boost_df[xgb_features]

    y = boost_df["ets_residual"]

    tscv = TimeSeriesSplit(n_splits=5)

    cv_wape = []

    best_estimators = []

    for fold, (tr_idx, val_idx) in enumerate(tscv.split(X)):

        X_tr = X.iloc[tr_idx]
        X_val = X.iloc[val_idx]

        y_tr = y.iloc[tr_idx]
        y_val = y.iloc[val_idx]

        model = XGBRegressor(
            n_estimators=2500,
            learning_rate=0.012,
            max_depth=5,
            subsample=0.85,
            colsample_bytree=0.85,
            min_child_weight=2,
            gamma=0.03,
            reg_alpha=0.1,
            reg_lambda=1.2,
            objective="reg:pseudohubererror",
            eval_metric="mae",
            random_state=42,
            n_jobs=-1,
            early_stopping_rounds=60,
        )

        model.fit(
            X_tr,
            y_tr,
            eval_set=[(X_val,y_val)],
            verbose=False,
        )

        best_estimators.append(
            model.best_iteration + 1
        )

        pred_res = model.predict(X_val)

        val_dates = X_val.index

        ets_pred = ets_fit.fittedvalues.loc[val_dates]

        hybrid = np.maximum(
            ets_pred.values + pred_res,
            0
        )

        real = df_working.loc[
            val_dates,
            "guests_count"
        ]

        fold_wape = wape(real, hybrid)

        cv_wape.append(fold_wape)

        print(
            f"fold {fold+1} | "
            f"WAPE={fold_wape:.3f}%"
        )

    print(
        f"\nCV WAPE: "
        f"{np.mean(cv_wape):.3f}%"
    )

    best_n = int(np.median(best_estimators))

    xgb_final = XGBRegressor(
        n_estimators=best_n,
        learning_rate=0.012,
        max_depth=5,
        subsample=0.85,
        colsample_bytree=0.85,
        min_child_weight=2,
        gamma=0.03,
        reg_alpha=0.1,
        reg_lambda=1.2,
        objective="reg:pseudohubererror",
        random_state=42,
        n_jobs=-1,
    )

    xgb_final.fit(X,y)

    importance = pd.DataFrame({
        "feature": xgb_features,
        "importance": xgb_final.feature_importances_,
    }).sort_values(
        "importance",
        ascending=False
    )

    return {
        "xgb": xgb_final,
        "ets_fit": ets_fit,
        "forecast_fe": forecast_fe,
        "forecast_index": forecast_index,
        "feature_cols": feature_cols,
        "xgb_features": xgb_features,
        "res_lag_cols": res_lag_cols,
        "res_scaler": res_scaler,
        "df_working": df_working,
        "importance": importance,
        "train_raw": train_raw,
    }


# ============================================================
# FORECAST
# ============================================================

def make_forecast(
    artifacts,
    holiday_set,
):

    xgb = artifacts["xgb"]

    ets_fit = artifacts["ets_fit"]

    forecast_fe = artifacts["forecast_fe"]

    forecast_index = artifacts["forecast_index"]

    feature_cols = artifacts["feature_cols"]

    xgb_features = artifacts["xgb_features"]

    res_lag_cols = artifacts["res_lag_cols"]

    res_scaler = artifacts["res_scaler"]

    train_raw = artifacts["train_raw"]

    ets_forecast = ets_fit.forecast(
        len(forecast_index)
    )

    X_fc = forecast_fe[feature_cols].copy()

    zero_df = pd.DataFrame(
        np.zeros((len(X_fc), len(res_lag_cols))),
        columns=res_lag_cols,
    )

    z_sc = res_scaler.transform(
        zero_df
    )

    for i,col in enumerate(res_lag_cols):

        X_fc[col] = z_sc[:,i]

    for col in xgb_features:

        if col not in X_fc.columns:
            X_fc[col] = 0

    res_pred = xgb.predict(
        X_fc[xgb_features]
    )

    ml_forecast = np.maximum(
        ets_forecast.values + res_pred,
        0
    )

    # ========================================================
    # ANALOG ENSEMBLE
    # ========================================================

    YOY_TREND = 1.067

    analog_values = []

    for ts in forecast_index:

        hour = ts.hour

        values = []

        # ====================================================
        # 2025 exact analog
        # ====================================================

        d2025 = str(ts.date()).replace(
            "2026",
            "2025"
        )

        m25 = train_raw[
            (train_raw["sale_date"] == pd.Timestamp(d2025))
            &
            (train_raw["sale_hour"] == hour)
        ]

        if len(m25):

            values.append(
                m25["guests_count"].iloc[0]
                * YOY_TREND
            )

        # ====================================================
        # same weekday analog
        # ====================================================

        weekday_match = train_raw[
            (train_raw["sale_hour"] == hour)
            &
            (
                train_raw["sale_date"].dt.dayofweek
                ==
                ts.dayofweek
            )
        ].tail(4)

        if len(weekday_match):

            values.append(
                weekday_match["guests_count"].mean()
            )

        if len(values):

            analog_values.append(
                np.mean(values)
            )

        else:
            analog_values.append(np.nan)

    analog_values = np.array(analog_values)

    final = ml_forecast.copy()

    mask = ~np.isnan(analog_values)

    # ========================================================
    # DYNAMIC ENSEMBLE
    # ========================================================

    alpha_ml = np.where(
        forecast_index.hour <= 10,
        0.25,
        0.45
    )

    alpha_analog = 1 - alpha_ml

    final[mask] = (
        alpha_ml[mask] * ml_forecast[mask]
        +
        alpha_analog[mask] * analog_values[mask]
    )

    # ========================================================
    # HOLIDAY CORRECTION
    # ========================================================

    holiday_hours = (
        train_raw.groupby("sale_hour")["guests_count"]
        .median()
    )

    for i, ts in enumerate(forecast_index):

        ds = ts.strftime("%Y-%m-%d")

        if ds in holiday_set:

            final[i] *= 0.82

        if ts.hour in HIGH_ERROR_HOURS:

            floor = holiday_hours[ts.hour] * 0.55

        else:

            floor = holiday_hours[ts.hour] * 0.40

        final[i] = max(final[i], floor)

    final = np.round(final).astype(int)

    result_std = pd.DataFrame({
        "sale_date": [
            str(ts.date())
            for ts in forecast_index
        ],
        "sale_hour": forecast_index.hour,
        "guests_count": final,
    })

    result_kaggle = pd.DataFrame({
        "ID": [
            f"{ts.date()}-{ts.hour:02d}"
            for ts in forecast_index
        ],
        "guests_count": final,
    })

    return result_std, result_kaggle


# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":

    os.makedirs("./output", exist_ok=True)

    train_raw = pd.read_csv(
        f"{DATA_DIR}/train.csv"
    )

    train_raw["sale_date"] = pd.to_datetime(
        train_raw["sale_date"]
    )

    train_raw = train_raw.loc[
        train_raw["sale_date"] >= CUT_DATE
    ].copy()

    print(
        f"Train after cut: {len(train_raw)} rows | "
        f"{train_raw['sale_date'].min().date()} – "
        f"{train_raw['sale_date'].max().date()}"
    )

    years = list(range(
        train_raw["sale_date"].dt.year.min(),
        2027
    ))

    holiday_set = fetch_holidays_api(
        years
    )

    weather_df = fetch_weather_api(
        train_raw["sale_date"].min().strftime("%Y-%m-%d"),
        FORECAST_END,
    )

    artifacts = build_model(
        train_raw,
        holiday_set,
        weather_df,
    )
    result_std, result_kaggle = make_forecast(
        artifacts,
        holiday_set,
    )

    result_std.to_excel(
        "./output/forecast_standard.xlsx",
        index=False
    )

    result_kaggle.to_csv(
        "./output/forecast_kaggle.csv",
        index=False
    )



    print("\nDONE")

    print("\nforecast_standard.xlsx")
    print(result_std.head())

    print("\nforecast_kaggle.csv")
    print(result_kaggle.head())

    print("\nTOP FEATURES")
    print(
        artifacts["importance"]
        .head(20)
        .to_string(index=False)
    )